# Experimento D: Pipeline Automatizado de Punta a Punta (3.5M Pasos)

Pipeline unificado de entrenamiento: ejecución del bucle PPO,
registro de métricas empíricas y serialización a formato ONNX.


### Artefactos de salida:
1. `modelo_magic_jewelry_experimento_d.onnx` (Modelo para la web)
2. `reporte_entrenamiento_experimento_d.csv` (Métricas de partidas)
3. `grafica_entrenamiento_experimento_d.png` (Curva de aprendizaje)

### 1. Instalación de Dependencias

In [ ]:
!pip install --quiet gymnasium stable-baselines3 onnx onnxruntime onnxscript pandas matplotlib

### 2. Entorno de Simulación v4 con Reward Shaping Calibrado

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np


class EntornoMagicJewelry(gym.Env):
    metadata = {"render_modes": ["ansi"]}

    FILAS = 13
    COLUMNAS = 6
    NUM_COLORES_ESTANDAR = 6
    COLOR_GEMA_MAGICA = 7
    PUNTOS_POR_GEMA = 30
    JOYAS_POR_NIVEL = 70
    UMBRAL_ALTURA_PELIGRO = 8

    def __init__(self):
        super().__init__()
        self.action_space = spaces.Discrete(18)
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(84,), dtype=np.float32
        )
        self.tablero = np.zeros((self.FILAS, self.COLUMNAS), dtype=np.int8)
        self.pieza_actual    = np.zeros(3, dtype=np.int8)
        self.pieza_siguiente = np.zeros(3, dtype=np.int8)
        self.gemas_destruidas = 0
        self.siguiente_umbral_magica = self.JOYAS_POR_NIVEL
        self.proxima_es_magica = False
        self.puntaje_acumulado = 0
        self.piezas_colocadas  = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.tablero.fill(0)
        self.puntaje_acumulado = 0
        self.piezas_colocadas  = 0
        self.gemas_destruidas  = 0
        self.siguiente_umbral_magica = self.JOYAS_POR_NIVEL
        self.proxima_es_magica = False
        self.pieza_actual    = self._generar_pieza_aleatoria()
        self.pieza_siguiente = self._generar_pieza_aleatoria()
        return self._obtener_observacion(), {}

    def step(self, action: int):
        columna  = int(action) // 3
        rotacion = int(action) % 3

        es_magica = (self.pieza_actual[0] == self.COLOR_GEMA_MAGICA)
        pieza_rotada = self._rotar_pieza(self.pieza_actual, rotacion) if not es_magica else self.pieza_actual.copy()
        tablero_previo = self.tablero.copy()

        filas_ocupadas, desborde = self._colocar_y_caer(columna, pieza_rotada)
        self.piezas_colocadas += 1

        bonus_adyacencia = 0.0
        if not es_magica and not desborde:
            conexiones = self._contar_conexiones_adyacentes(filas_ocupadas, columna, tablero_previo)
            bonus_adyacencia = conexiones * (self.PUNTOS_POR_GEMA * 0.25 * 0.1)

        if es_magica:
            gemas_eliminadas, puntos_turno = self._procesar_impacto_magica(filas_ocupadas, columna)
        else:
            gemas_eliminadas, puntos_turno = self._resolver_cascadas()

        terminado = False
        if desborde and gemas_eliminadas == 0:
            terminado = True
        elif self.tablero[0, 3] != 0:
            terminado = True

        recompensa = self._calcular_recompensa_v4(
            puntos_turno=puntos_turno,
            bonus_adyacencia=bonus_adyacencia,
            terminado=terminado
        )

        self.puntaje_acumulado += puntos_turno
        self.gemas_destruidas  += gemas_eliminadas

        if self.gemas_destruidas >= self.siguiente_umbral_magica:
            self.proxima_es_magica = True
            while self.siguiente_umbral_magica <= self.gemas_destruidas:
                self.siguiente_umbral_magica += self.JOYAS_POR_NIVEL

        if not terminado:
            self.pieza_actual = self.pieza_siguiente.copy()
            if self.proxima_es_magica:
                self.pieza_siguiente = np.array([self.COLOR_GEMA_MAGICA] * 3, dtype=np.int8)
                self.proxima_es_magica = False
            else:
                self.pieza_siguiente = self._generar_pieza_aleatoria()

        info = {
            "puntaje": self.puntaje_acumulado,
            "piezas": self.piezas_colocadas,
            "gemas_destruidas": self.gemas_destruidas,
            "es_magica": es_magica
        }
        return self._obtener_observacion(), recompensa, terminado, False, info

    def _calcular_recompensa_v4(self, puntos_turno: int, bonus_adyacencia: float, terminado: bool) -> float:
        if terminado:
            penalizacion = 15.0 + 0.3 * self.piezas_colocadas
            return -min(penalizacion, 40.0)

        recompensa = (puntos_turno * 0.1) + bonus_adyacencia + 1.0

        alturas = [
            self.FILAS - np.argmax(self.tablero[:, c] != 0)
            if np.any(self.tablero[:, c] != 0) else 0
            for c in range(self.COLUMNAS)
        ]
        altura_maxima = max(alturas)

        if altura_maxima > self.UMBRAL_ALTURA_PELIGRO:
            exceso = (altura_maxima - self.UMBRAL_ALTURA_PELIGRO) / (self.FILAS - self.UMBRAL_ALTURA_PELIGRO)
            recompensa -= exceso * exceso * 3.0

        return float(recompensa)

    def _resolver_cascadas(self):
        total_gemas    = 0
        puntos_totales = 0
        ronda = 0

        while True:
            matches = self._buscar_matches()
            if not matches:
                break
            ronda += 1
            cantidad = len(matches)
            total_gemas += cantidad
            multiplicador = 1.3 ** (ronda - 1)
            puntos_totales += int(cantidad * self.PUNTOS_POR_GEMA * multiplicador)
            for f, c in matches:
                self.tablero[f, c] = 0
            self._aplicar_gravedad()

        return total_gemas, puntos_totales

    def _generar_pieza_aleatoria(self) -> np.ndarray:
        return self.np_random.integers(1, self.NUM_COLORES_ESTANDAR + 1, size=3, dtype=np.int8)

    def _rotar_pieza(self, pieza: np.ndarray, rotacion: int) -> np.ndarray:
        if rotacion == 0:
            return pieza.copy()
        elif rotacion == 1:
            return np.array([pieza[2], pieza[0], pieza[1]], dtype=np.int8)
        else:
            return np.array([pieza[1], pieza[2], pieza[0]], dtype=np.int8)

    def _colocar_y_caer(self, columna: int, pieza: np.ndarray):
        fila_primer_bloque = self.FILAS
        for f in range(self.FILAS):
            if self.tablero[f, columna] != 0:
                fila_primer_bloque = f
                break
        fila_asiento = fila_primer_bloque - 1
        desborde = False
        filas_ocupadas = []
        for i in range(3):
            fila_destino = fila_asiento - i
            gema = pieza[2 - i]
            if fila_destino >= 0:
                self.tablero[fila_destino, columna] = gema
                filas_ocupadas.append(fila_destino)
            else:
                desborde = True
        return filas_ocupadas, desborde

    def _contar_conexiones_adyacentes(self, filas_ocupadas: list, columna: int, tablero_previo: np.ndarray) -> int:
        conexiones = 0
        dirs = [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]
        for f in filas_ocupadas:
            color = self.tablero[f, columna]
            if color == 0:
                continue
            for df, dc in dirs:
                nf, nc = f + df, columna + dc
                if 0 <= nf < self.FILAS and 0 <= nc < self.COLUMNAS:
                    if tablero_previo[nf, nc] == color:
                        conexiones += 1
        return conexiones

    def _procesar_impacto_magica(self, filas_ocupadas: list, columna: int):
        if not filas_ocupadas:
            return 0, 0
        fila_asiento = max(filas_ocupadas)
        fila_debajo  = fila_asiento + 1
        gemas_eliminadas_color = 0
        if fila_debajo < self.FILAS:
            color_objetivo = self.tablero[fila_debajo, columna]
            if color_objetivo != 0 and color_objetivo != self.COLOR_GEMA_MAGICA:
                mascara = (self.tablero == color_objetivo)
                gemas_eliminadas_color = int(np.sum(mascara))
                self.tablero[mascara] = 0
        self.tablero[self.tablero == self.COLOR_GEMA_MAGICA] = 0
        self._aplicar_gravedad()
        total_destruidas = gemas_eliminadas_color + 3
        puntos_magica    = total_destruidas * self.PUNTOS_POR_GEMA
        gemas_cascadas, puntos_cascadas = self._resolver_cascadas()
        return (total_destruidas + gemas_cascadas), (puntos_magica + puntos_cascadas)

    def _buscar_matches(self) -> set:
        gemas_a_eliminar = set()
        for f in range(self.FILAS):
            for c in range(self.COLUMNAS):
                color = self.tablero[f, c]
                if color == 0 or color == self.COLOR_GEMA_MAGICA:
                    continue
                if c + 2 < self.COLUMNAS and color == self.tablero[f, c+1] == self.tablero[f, c+2]:
                    gemas_a_eliminar.update({(f,c),(f,c+1),(f,c+2)})
                if f + 2 < self.FILAS and color == self.tablero[f+1, c] == self.tablero[f+2, c]:
                    gemas_a_eliminar.update({(f,c),(f+1,c),(f+2,c)})
                if f+2 < self.FILAS and c+2 < self.COLUMNAS and color == self.tablero[f+1,c+1] == self.tablero[f+2,c+2]:
                    gemas_a_eliminar.update({(f,c),(f+1,c+1),(f+2,c+2)})
                if f+2 < self.FILAS and c-2 >= 0 and color == self.tablero[f+1,c-1] == self.tablero[f+2,c-2]:
                    gemas_a_eliminar.update({(f,c),(f+1,c-1),(f+2,c-2)})
        return gemas_a_eliminar

    def _aplicar_gravedad(self):
        for c in range(self.COLUMNAS):
            col_datos   = self.tablero[:, c]
            gemas_vivas = col_datos[col_datos != 0]
            ceros = self.FILAS - len(gemas_vivas)
            self.tablero[:ceros, c] = 0
            self.tablero[ceros:, c] = gemas_vivas

    def _obtener_observacion(self) -> np.ndarray:
        tablero_norm   = (self.tablero.flatten() / float(self.COLOR_GEMA_MAGICA)).astype(np.float32)
        actual_norm    = (self.pieza_actual    / float(self.COLOR_GEMA_MAGICA)).astype(np.float32)
        siguiente_norm = (self.pieza_siguiente / float(self.COLOR_GEMA_MAGICA)).astype(np.float32)
        return np.concatenate([tablero_norm, actual_norm, siguiente_norm])

print("Entorno v4 definido exitosamente.")

### 3. Definición de Callbacks y Exportador ONNX

In [ ]:
import os, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from stable_baselines3.common.callbacks import BaseCallback
import torch
import torch.nn as nn


class CallbackMetricasD(BaseCallback):
    def __init__(self, frecuencia_reporte: int = 200, carpeta_salida: str = "/kaggle/working"):
        super().__init__()
        self.frecuencia_reporte = frecuencia_reporte
        self.carpeta_salida = carpeta_salida
        self.historial = []
        self.partida_num = 0
        self.tiempo_inicio = None

    def _on_training_start(self):
        self.tiempo_inicio = time.time()
        print("Entrenamiento iniciado. Registrando métricas...")

    def _on_step(self) -> bool:
        dones = self.locals.get("dones", [])
        infos = self.locals.get("infos", [])
        for done, info in zip(dones, infos):
            if done:
                self.partida_num += 1
                self.historial.append({
                    "partida":          self.partida_num,
                    "paso_global":      self.num_timesteps,
                    "puntaje":          info.get("puntaje", 0),
                    "piezas_colocadas": info.get("piezas",  0),
                    "gemas_destruidas": info.get("gemas_destruidas", 0),
                })
                if self.partida_num % self.frecuencia_reporte == 0:
                    ultimas = self.historial[-self.frecuencia_reporte:]
                    df_tmp  = pd.DataFrame(ultimas)
                    minutos = (time.time() - self.tiempo_inicio) / 60
                    print(
                        f"[D] Partida {self.partida_num:>6} | "
                        f"Paso {self.num_timesteps:>9,} | "
                        f"Pts media: {df_tmp['puntaje'].mean():>7.1f} | "
                        f"Pts max: {df_tmp['puntaje'].max():>6} | "
                        f"Gemas max: {df_tmp['gemas_destruidas'].max():>4} | "
                        f"Tiempo: {minutos:.1f}m"
                    )
        return True

    def guardar_reportes(self):
        df = pd.DataFrame(self.historial)
        if df.empty:
            df = pd.DataFrame([{"partida": 0, "paso_global": 0, "puntaje": 0, "piezas_colocadas": 0, "gemas_destruidas": 0}])

        ruta_csv = os.path.join(self.carpeta_salida, "reporte_entrenamiento_experimento_d.csv")
        df.to_csv(ruta_csv, index=False)
        print(f"✓ CSV guardado: '{ruta_csv}' ({len(df):,} partidas)")

        try:
            fig, axes = plt.subplots(3, 1, figsize=(12, 9))
            fig.suptitle("Experimento D: Reward Shaping Calibrado (3.5M Pasos)", fontsize=14, fontweight="bold")
            cols = ["puntaje", "piezas_colocadas", "gemas_destruidas"]
            colores = ["#e63946", "#2a9d8f", "#8338ec"]
            ylabels = ["Puntos", "Piezas colocadas", "Gemas eliminadas"]

            for ax, col, color, ylabel in zip(axes, cols, colores, ylabels):
                ax.plot(df[col], alpha=0.2, color="#888888")
                ax.plot(df[col].rolling(50, min_periods=1).mean(), color=color, linewidth=2, label="Media móvil (50 eps)")
                ax.set_ylabel(ylabel)
                ax.legend()
                ax.grid(True, alpha=0.3)

            axes[2].set_xlabel("Número de partida (Episodio)")
            plt.tight_layout()
            ruta_png = os.path.join(self.carpeta_salida, "grafica_entrenamiento_experimento_d.png")
            plt.savefig(ruta_png, dpi=200)
            plt.close()
            print(f"✓ Gráfica guardada: '{ruta_png}'")
        except Exception as e:
            print(f"[Aviso gráfica] {e}")


class PoliticaActorONNX(nn.Module):
    def __init__(self, policy):
        super().__init__()
        self.mlp_extractor = policy.mlp_extractor.policy_net.to("cpu")
        self.action_net    = policy.action_net.to("cpu")

    def forward(self, x):
        return torch.argmax(self.action_net(self.mlp_extractor(x)), dim=-1)


def exportar_modelo_a_onnx(modelo_ppo, ruta_destino: str):
    # Exportación ONNX mediante PyTorch nativo
    modelo_ppo.policy.to("cpu")
    red_actor = PoliticaActorONNX(modelo_ppo.policy)
    red_actor.eval()

    tensor_ejemplo = torch.zeros((1, 84), dtype=torch.float32, device="cpu")
    torch.onnx.export(
        red_actor,
        tensor_ejemplo,
        ruta_destino,
        input_names=["observacion"],
        output_names=["accion"],
        opset_version=18
    )
    print(f"✓ ONNX exportado exitosamente con PyTorch: '{ruta_destino}'")

    try:
        import onnxruntime as ort
        sesion = ort.InferenceSession(ruta_destino)
        prueba = sesion.run(None, {"observacion": np.zeros((1, 84), dtype=np.float32)})
        print(f"✓ Inferencia verificada: Acción de prueba = {prueba[0][0]}")
    except Exception:
        pass

print("Callback y exportador ONNX listos.")

### 4. Entrenamiento y Exportación Total Automatizada

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
import os

NUM_ENTORNOS = 8
PASOS_TOTALES = 3_500_000
CARPETA_SALIDA = "/kaggle/working"

print(f"Configurando entorno con {NUM_ENTORNOS} instancias...")
env_vectorizado = make_vec_env(EntornoMagicJewelry, n_envs=NUM_ENTORNOS)

policy_kwargs = dict(
    net_arch=dict(pi=[128, 128, 128], vf=[128, 128, 128])
)

modelo_d = PPO(
    policy="MlpPolicy",
    env=env_vectorizado,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=128,
    n_epochs=10,
    gamma=0.99,
    ent_coef=0.01,
    policy_kwargs=policy_kwargs,
    verbose=0
)

callback_d = CallbackMetricasD(frecuencia_reporte=200, carpeta_salida=CARPETA_SALIDA)

print(f"\n--- INICIANDO ENTRENAMIENTO COMPLETO ({PASOS_TOTALES:,} PASOS) ---")
modelo_d.learn(total_timesteps=PASOS_TOTALES, callback=callback_d)
print("--- ENTRENAMIENTO FINALIZADO ---\n")

print("Generando todos los archivos de salida...")

# 1. Exportar ONNX (Usa solo PyTorch nativo)
ruta_onnx = os.path.join(CARPETA_SALIDA, "modelo_magic_jewelry_experimento_d.onnx")
exportar_modelo_a_onnx(modelo_d, ruta_onnx)

# 2. Guardar métricas CSV y Gráfica PNG
callback_d.guardar_reportes()

# 3. Guardar checkpoint PyTorch
modelo_d.save(os.path.join(CARPETA_SALIDA, "magic_jewelry_ppo_experimento_d_final"))

print("\n========================================================")
print("Pipeline finalizado. Artefactos exportados:")
print("  1. modelo_magic_jewelry_experimento_d.onnx")
print("  2. reporte_entrenamiento_experimento_d.csv")
print("  3. grafica_entrenamiento_experimento_d.png")
print("========================================================")